# YOROSIX 行動データパイプライン v10（多次元アクション対応）

YOROSIXの協力行動データを、**フィジカルAI / ソーシャルシミュレーション / 行動デジタルツイン(BDT)** が消費できる形で出力するパイプライン。

## v10の拡張: 多次元アクション
行動を離散3択(none/reply/share)から、**実行有無 + 4つの連続次元**に拡張:
- **intensity** … コミット度(「やる」vs「できるかも」)
- **effort** … 返信の具体性/労力
- **speed** … 応答の速さ(熱量の代理)
- **content_topic** … 返信の内容カテゴリ

→ 「リクエストへの返信は2択やなく無限にある」を連続次元として表現。BDTは「行動するか」だけでなく「どんな返信をするか(強度・労力・速さ)」まで予測する。

## 3性質(維持)
- 非行動が多数派 / 文脈依存 / ラベル独立(リーク無し) + 反実仮想介入

## データ源
- 合成生成器(中身を意味ある形で生成) と 実ログ取り込み が**同一スキーマ**を出力
- 実データの返信内容はまだ薄いため、多次元の中身は合成で実証、実データは将来流入の受け皿


## 0. セットアップ

In [ ]:
!pip install pandas numpy scikit-learn pyarrow networkx pyyaml

In [ ]:
import pandas as pd, numpy as np, json, pickle, bisect, glob, os
from datetime import datetime, timedelta
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional, List

## 1. 標準行動イベントスキーマ（多次元アクション）

In [ ]:
"""
core/schema.py
========================================================
標準行動イベントスキーマ — 全ダウンストリーム用途の唯一の源。

フィジカルAI / ソーシャルシミュ / BDT は、すべてこの BehaviorEvent
テーブルから派生する。スキーマを1箇所に固定することで、3用途が
同じパイプラインから出力できる(= データパイプラインを永久資産にする)。

リーク防止の不変条件:
  state_features は必ず timestamp 以前の情報のみ。
  outcome(response/thanks) は行動の「下流」であり、特徴量には絶対に入れない。
"""

from dataclasses import dataclass, field, asdict
from typing import Optional, List
import pandas as pd

# 行動の語彙(全用途共通)
ACTIONS = ["none", "reply", "share"]
ACTION_TO_ID = {a: i for i, a in enumerate(ACTIONS)}


# state_features に入れてよい列(露出時点で観測可能・リーク無し)
STATE_FEATURE_COLS = [
    # 人物固有(時間非依存)
    "u_influence", "u_latent_coop", "u_activity", "u_netpos",
    "u_sens_urgency", "u_sens_tie", "u_sens_time", "u_sens_load",
    # T以前の累積行動
    "past_shares", "past_replies", "past_exposures", "past_conversion",
    "past_act_this_topic", "topic_recency_h", "topic_affinity",
    # リクエスト文脈
    "req_urgency", "req_text_quality", "time_match", "topic_fit",
    # ネットワーク関係
    "tie_strength", "same_cluster",
]

# state_features に入れてはいけない列(下流・リーク源)
FORBIDDEN_IN_STATE = [
    "past_thx_received", "past_thx_given",   # thanks は行動の下流
    "outcome_response", "outcome_thanks",
    "action", "label_act",
]


@dataclass
class BehaviorEvent:
    """単一の行動イベント。全用途がこれを変換して使う。"""
    event_id: int
    timestamp: str
    actor_id: int
    request_id: int
    # 状況(行動直前・リーク無し)
    state: dict
    # 行動(type)
    action: str                      # none | reply | share
    # --- 多次元アクション(行動の中身。none のときは既定値) ---
    intensity: float = 0.0           # コミット度 0..1 (「やる」vs「できるかも」)
    content_topic: int = -1          # 返信の内容カテゴリ(どんな助けか)
    speed: float = 0.0               # 応答の速さ 0..1(速い=熱量の代理)
    effort: float = 0.0              # 返信の具体性/長さスコア 0..1
    # 下流の結果(因果順で行動の後)
    outcome_response: int = 0        # 依頼者から応答があったか
    outcome_thanks: int = 0          # thanks を受け取ったか
    # ネットワーク文脈
    tie_strength: float = 0.0
    netpos: float = 0.0
    cluster_id: int = -1

    def validate(self):
        """リーク不変条件をチェック。"""
        for f in FORBIDDEN_IN_STATE:
            if f in self.state:
                raise ValueError(f"LEAK: '{f}' は state に含めてはいけない")
        if self.action not in ACTIONS:
            raise ValueError(f"unknown action: {self.action}")
        # 多次元アクションの整合性: none のときは中身ゼロ
        if self.action == "none" and (self.intensity > 0 or self.effort > 0):
            raise ValueError("non-action なのに intensity/effort が非ゼロ")
        return True


def events_to_frame(events: List[BehaviorEvent]) -> pd.DataFrame:
    """BehaviorEvent のリストを平坦な DataFrame に。"""
    rows = []
    for e in events:
        row = {
            "event_id": e.event_id, "timestamp": e.timestamp,
            "actor_id": e.actor_id, "request_id": e.request_id,
            "action": e.action, "action_id": ACTION_TO_ID[e.action],
            # 多次元アクション
            "act_intensity": e.intensity, "act_content_topic": e.content_topic,
            "act_speed": e.speed, "act_effort": e.effort,
            "outcome_response": e.outcome_response,
            "outcome_thanks": e.outcome_thanks,
            "tie_strength": e.tie_strength, "netpos": e.netpos,
            "cluster_id": e.cluster_id,
        }
        for k, v in e.state.items():
            row[f"state__{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


def state_columns(df: pd.DataFrame) -> List[str]:
    """DataFrame から state__ 接頭辞の列を返す。"""
    return [c for c in df.columns if c.startswith("state__")]


def assert_no_leak(df: pd.DataFrame):
    """DataFrame レベルでリーク列が state に混入してないか検査。"""
    for c in state_columns(df):
        bare = c.replace("state__", "")
        if bare in FORBIDDEN_IN_STATE:
            raise ValueError(f"LEAK detected in dataframe: {c}")
    return True


## 2. 合成生成器（3性質 + 多次元アクションを文脈依存で生成）

In [ ]:
"""
core/world.py
========================================================
v8 生成器(3性質)をクラス化し、標準 BehaviorEvent を直接吐く。

3性質:
  (1) 露出 >> 行動  … exposed 全員を候補にし、行動有無を別サンプリング
  (2) P(行動) = f(人物, 文脈, ネット位置, 状態)
  (3) ラベル独立 / 因果順 expose -> act -> response -> thanks

state は露出時点で構築(T以前の累積のみ)。リークが構造的に入らない。
"""

import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
import bisect



def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


class YorosixWorld:
    def __init__(self, cfg):
        self.cfg = cfg
        w = cfg["world"]
        self.n_members = w["n_members"]
        self.n_posts = w["n_posts"]
        self.seed = w["seed"]
        self.n_clusters = w["n_clusters"]
        self.n_topics = w["n_topics"]
        self.G = cfg["generation"]
        self.base_time = datetime(2024, 1, 1)
        self._build_members()

    def _build_members(self):
        rng = np.random.default_rng(self.seed)
        n = self.n_members
        self.rng = rng
        self.mem = {
            "id": np.arange(1, n + 1),
            "influence": rng.pareto(2.0, n) + 1.0,
            "latent_coop": rng.normal(0, 1, n),
            "cluster_id": rng.integers(0, self.n_clusters, n),
            "bridge": rng.binomial(1, 0.08, n),
            "activity": rng.beta(2, 4, n),
            "home_topic": rng.integers(0, self.n_topics, n),
            "sens_urgency": rng.normal(1.0, 0.6, n),
            "sens_tie": rng.normal(1.0, 0.5, n),
            "sens_time": rng.normal(1.0, 0.4, n),
            "sens_load": rng.uniform(0.5, 1.5, n),
        }
        # 索引
        self.idx = {mid: i for i, mid in enumerate(self.mem["id"])}
        # クラスタ/bridge
        self.cluster_members = {
            c: self.mem["id"][self.mem["cluster_id"] == c]
            for c in range(self.n_clusters)}
        self.bridges = self.mem["id"][self.mem["bridge"] == 1]
        # degree / netpos
        deg = {}
        for mid in self.mem["id"]:
            i = self.idx[mid]
            c = self.mem["cluster_id"][i]
            d = len(self.cluster_members[c]) - 1
            if self.mem["bridge"][i] == 1:
                d += len(self.bridges)
            deg[mid] = d
        mx = max(deg.values())
        self.netpos = {k: v / mx for k, v in deg.items()}

    def _m(self, mid, key):
        return self.mem[key][self.idx[mid]]

    def generate(self, urgency_shift=0.0):
        """BehaviorEvent のリストを返す。urgency_shift で反実仮想世界。"""
        rng = np.random.default_rng(self.seed + (1 if urgency_shift else 0))
        G = self.G

        # 状態(時間発展): 負荷, 個人の過去行動時刻
        load = {mid: 0.0 for mid in self.mem["id"]}
        last_t = {mid: self.base_time for mid in self.mem["id"]}
        # 累積履歴(state構築用): user -> {type -> [times]}, user -> topic -> [times]
        hist = defaultdict(lambda: defaultdict(list))
        hist_topic = defaultdict(lambda: defaultdict(list))
        exp_count = defaultdict(int)

        events = []
        eid = 1

        for p in range(self.n_posts):
            post_time = self.base_time + timedelta(minutes=p * 15)
            infl = self.mem["influence"]
            origin = int(rng.choice(self.mem["id"], p=infl / infl.sum()))
            o_cluster = self._m(origin, "cluster_id")
            urgency = float(np.clip(rng.uniform(0.1, 0.95) + urgency_shift, 0, 1))
            topic = int(rng.integers(0, self.n_topics))
            text_q = float(rng.uniform(0.2, 1.0))
            hour = post_time.hour

            neigh = np.unique(np.concatenate(
                [self.cluster_members[o_cluster], self.bridges]))
            neigh = neigh[neigh != origin]

            for u in neigh:
                if rng.random() >= 0.25 * self._m(u, "activity") + 0.05:
                    continue  # 露出しない
                u = int(u)
                T = post_time + timedelta(minutes=int(rng.integers(1, 120)))

                # --- state を T 以前の情報のみで構築(リーク無し) ---
                dt_h = (post_time - last_t[u]).total_seconds() / 3600.0
                load[u] *= np.exp(-dt_h / 12.0)
                same_c = (self._m(u, "cluster_id") == o_cluster)
                tie = 1.0 if same_c else (0.4 if self._m(u, "bridge") else 0.2)
                time_match = np.cos((hour - 12) / 24 * 2 * np.pi) * 0.5 + 0.5
                topic_fit = 1.0 if self._m(u, "home_topic") == topic else 0.0

                past_share = len(hist[u]["share"])
                past_reply = len(hist[u]["reply"])
                past_exp = exp_count[u]
                past_act = past_share + past_reply
                past_conv = past_act / past_exp if past_exp > 0 else 0.0
                tphist = hist_topic[u][topic]
                past_act_topic = len(tphist)
                recency_h = ((T - tphist[-1]).total_seconds() / 3600.0
                             if tphist else 720.0)
                topic_aff = past_act_topic / (past_act + 1)

                state = {
                    "u_influence": self._m(u, "influence"),
                    "u_latent_coop": self._m(u, "latent_coop"),
                    "u_activity": self._m(u, "activity"),
                    "u_netpos": self.netpos[u],
                    "u_sens_urgency": self._m(u, "sens_urgency"),
                    "u_sens_tie": self._m(u, "sens_tie"),
                    "u_sens_time": self._m(u, "sens_time"),
                    "u_sens_load": self._m(u, "sens_load"),
                    "past_shares": past_share, "past_replies": past_reply,
                    "past_exposures": past_exp, "past_conversion": past_conv,
                    "past_act_this_topic": past_act_topic,
                    "topic_recency_h": min(recency_h, 720.0),
                    "topic_affinity": topic_aff,
                    "req_urgency": urgency, "req_text_quality": text_q,
                    "time_match": time_match, "topic_fit": topic_fit,
                    "tie_strength": tie, "same_cluster": int(same_c),
                }

                # --- 行動サンプリング(真の生成式) ---
                z = (G["beta0_share"]
                     + G["w_latent_coop"] * self._m(u, "latent_coop")
                     + G["w_urgency"] * self._m(u, "sens_urgency") * urgency
                     + G["w_tie"] * self._m(u, "sens_tie") * tie
                     + G["w_time"] * self._m(u, "sens_time") * time_match
                     - G["w_load"] * self._m(u, "sens_load") * load[u]
                     + G["w_netpos"] * self.netpos[u]
                     + G["w_topic"] * topic_fit
                     + rng.normal(0, G["noise_sd"]))
                p_share = sigmoid(z)
                z_reply = z - G["beta0_share"] + G["beta0_reply"]
                p_reply = sigmoid(z_reply)

                did_share = rng.random() < p_share
                did_reply = (not did_share) and (rng.random() < p_reply)
                action = "share" if did_share else ("reply" if did_reply else "none")

                # --- 多次元アクション(行動した場合のみ意味を持つ) ---
                intensity = content_topic = speed = effort = 0.0
                content_topic = -1
                if action != "none":
                    # intensity: コミット度。緊急度・紐帯・潜在協力性 + ノイズ
                    #   = 「同じ人でも状況で返信の強さが変わる」(指摘の核)
                    z_int = (-1.2
                             + 2.2 * urgency
                             + 1.2 * tie
                             + 0.7 * self._m(u, "latent_coop")
                             + rng.normal(0, 0.4))
                    intensity = float(np.clip(sigmoid(z_int), 0.02, 0.99))
                    # content_topic: 個人のhome_topic と request topic の混合
                    content_topic = int(topic if rng.random() < 0.6
                                        else self._m(u, "home_topic"))
                    # speed: 速さ=熱量。intensity が高いほど速い + ノイズ
                    speed = float(np.clip(0.3 + 0.6 * intensity
                                          + rng.normal(0, 0.15), 0.01, 0.99))
                    # effort: 具体性/長さ。share より reply の方が高くなりやすい
                    base_effort = 0.65 if did_reply else 0.35
                    effort = float(np.clip(base_effort * (0.5 + intensity)
                                           + rng.normal(0, 0.12), 0.01, 0.99))

                # --- 下流(因果順) ---
                resp = thx = 0
                if action != "none":
                    if rng.random() < (0.6 if did_reply else 0.4):
                        thx = 1
                    resp = 1
                    hist[u][action].append(post_time)
                    hist_topic[u][topic].append(post_time)
                    load[u] += 1.0 if did_share else 0.6
                    last_t[u] = post_time
                exp_count[u] += 1

                ev = BehaviorEvent(
                    event_id=eid, timestamp=str(T), actor_id=u,
                    request_id=20000 + p, state=state, action=action,
                    intensity=round(intensity, 4), content_topic=content_topic,
                    speed=round(speed, 4), effort=round(effort, 4),
                    outcome_response=resp, outcome_thanks=thx,
                    tie_strength=round(tie, 3), netpos=round(self.netpos[u], 4),
                    cluster_id=int(self._m(u, "cluster_id")),
                )
                ev.validate()
                events.append(ev)
                eid += 1

        return events

## 3. 実データ取り込み（本番CSV → 標準スキーマ）

In [ ]:
"""
core/ingest_real.py
========================================================
YOROSIX 本番エクスポート CSV を標準 BehaviorEvent スキーマに変換する入口。

入力(本番DBエクスポート):
  posts_export, share_links_export, replies_export,
  request_visits_export, thanks_logs_export,
  request_metadata_export, members_export

変換ルール:
  - 候補集合 = request_visits(露出した全セッション)  → 非行動層を保持
  - action  = そのvisitが reply / share に至ったか(action_taken と
              share_links/replies の突合で判定)、それ以外は none
  - state   = visit の created_at(=T)以前の、その visitor の累積行動のみ
              (リーク防止: 合成版 world.py と同一の不変条件)
  - outcome = thanks_logs(行動の下流)
  - ripple_depth, channel は実データ固有の貴重なネットワーク文脈として保持

合成版と同じ events.parquet を吐く → 後段の3エクスポータは無改変で動く。
"""

import pandas as pd
import numpy as np
import glob
import os
import bisect
from collections import defaultdict



def _latest(prefix, data_dir):
    files = sorted(glob.glob(os.path.join(data_dir, f"{prefix}_*.csv")))
    if not files:
        files = sorted(glob.glob(os.path.join(data_dir, f"{prefix}.csv")))
    return files[-1] if files else None


def _read(prefix, data_dir, **kw):
    p = _latest(prefix, data_dir)
    if p is None:
        return None
    return pd.read_csv(p, **kw)


def ingest(data_dir, out_path=None):
    visits = _read("request_visits_export", data_dir)
    shares = _read("share_links_export", data_dir)
    replies = _read("replies_export", data_dir)
    thanks = _read("thanks_logs_export", data_dir)
    meta = _read("request_metadata_export", data_dir)
    posts = _read("posts_export", data_dir)

    if visits is None or len(visits) == 0:
        raise ValueError("request_visits が空: 露出ログが無いと候補集合を作れない")

    # タイムスタンプ正規化
    visits["created_at"] = pd.to_datetime(visits["created_at"], errors="coerce")
    if shares is not None and len(shares):
        shares["created_at"] = pd.to_datetime(shares["created_at"], errors="coerce")
    if replies is not None and len(replies):
        replies["created_at"] = pd.to_datetime(replies["created_at"], errors="coerce")
    if thanks is not None and len(thanks):
        thanks["created_at"] = pd.to_datetime(thanks["created_at"], errors="coerce")

    # visitor が NULL の行(匿名)は actor 不明 → 除外(BDT/RLに使えない)
    visits = visits.dropna(subset=["visitor_user_id"]).copy()
    visits["visitor_user_id"] = visits["visitor_user_id"].astype(int)

    # share/reply を (request_id, user) のセットに(行動ラベル用)
    share_set = set()
    if shares is not None and len(shares):
        shares["owner_user_id"] = shares["owner_user_id"].astype("Int64")
        for _, r in shares.dropna(subset=["owner_user_id"]).iterrows():
            share_set.add((r["request_id"], int(r["owner_user_id"])))
    reply_set = set()
    if replies is not None and len(replies) and "member_id" in replies.columns:
        for _, r in replies.dropna(subset=["member_id"]).iterrows():
            reply_set.add((r["request_id"], int(r["member_id"])))

    # thanks を下流フラグに
    thx_set = set()
    if thanks is not None and len(thanks):
        for _, r in thanks.iterrows():
            if pd.notna(r.get("giver_member_id")):
                thx_set.add((r["request_id"], int(r["giver_member_id"])))

    # meta を request_id 索引に
    meta_idx = {}
    if meta is not None and len(meta):
        for _, r in meta.iterrows():
            meta_idx[r["request_id"]] = r

    # 各 visitor の行動時刻リスト(state構築・リーク防止用)
    uet = defaultdict(lambda: defaultdict(list))
    if shares is not None and len(shares):
        for _, r in shares.dropna(subset=["owner_user_id"]).iterrows():
            uet[int(r["owner_user_id"])]["share"].append(r["created_at"])
    if replies is not None and len(replies) and "member_id" in replies.columns:
        for _, r in replies.dropna(subset=["member_id"]).iterrows():
            uet[int(r["member_id"])]["reply"].append(r["created_at"])
    for _, r in visits.iterrows():
        uet[int(r["visitor_user_id"])]["visit"].append(r["created_at"])
    for u in uet:
        for et in uet[u]:
            uet[u][et] = sorted([t for t in uet[u][et] if pd.notna(t)])

    def cnt_before(lst, T):
        return bisect.bisect_left(lst, T)

    events = []
    eid = 1
    for _, v in visits.sort_values("created_at").iterrows():
        uid = int(v["visitor_user_id"])
        rid = v["request_id"]
        T = v["created_at"]
        if pd.isna(T):
            continue

        # 行動判定(因果順: visit→action)
        if (rid, uid) in share_set:
            action = "share"
        elif (rid, uid) in reply_set:
            action = "reply"
        elif str(v.get("action_taken", "")) == "share":
            action = "share"
        elif str(v.get("action_taken", "")) == "reply":
            action = "reply"
        else:
            action = "none"

        # state(T以前のみ)
        et = uet.get(uid, {})
        past_share = cnt_before(et.get("share", []), T)
        past_reply = cnt_before(et.get("reply", []), T)
        past_visit = cnt_before(et.get("visit", []), T)
        past_act = past_share + past_reply
        past_conv = past_act / past_visit if past_visit > 0 else 0.0

        m = meta_idx.get(rid)
        urgency = float(m["urgency_level"]) / 5.0 if (m is not None and pd.notna(m.get("urgency_level"))) else 0.5
        text_len = float(m["text_length"]) if (m is not None and pd.notna(m.get("text_length"))) else 0.0

        dwell = float(v["dwell_time_seconds"]) if pd.notna(v.get("dwell_time_seconds")) else 0.0
        scroll = float(v["scrolled_percentage"]) if pd.notna(v.get("scrolled_percentage")) else 0.0
        ripple_depth = int(v["ripple_depth"]) if pd.notna(v.get("ripple_depth")) else 1

        state = {
            "past_shares": past_share,
            "past_replies": past_reply,
            "past_exposures": past_visit,
            "past_conversion": past_conv,
            "req_urgency": urgency,
            "req_text_length": text_len,
            "visit_dwell": dwell,            # 露出時点で観測可能(その閲覧の滞在)
            "visit_scroll": scroll,
            "ripple_depth": ripple_depth,    # ネットワーク文脈(実データ固有)
            "clicked_reply_button": int(v.get("clicked_reply_button", 0) or 0),
        }
        # リーク列が混入してないか
        for f in FORBIDDEN_IN_STATE:
            state.pop(f, None)

        ev = BehaviorEvent(
            event_id=eid, timestamp=str(T), actor_id=uid, request_id=int(rid),
            state=state, action=action,
            # 多次元アクション: 実データの返信内容がまだ薄いため neutral 値。
            # 将来 reply_text が充実したら intensity=具体性, content_topic=分類 で埋める。
            intensity=(0.5 if action != "none" else 0.0),
            content_topic=(int(m["topic"]) if (action != "none" and m is not None
                           and pd.notna(m.get("topic"))) else -1),
            speed=0.5 if action != "none" else 0.0,
            effort=0.5 if action != "none" else 0.0,
            outcome_response=int(action != "none"),
            outcome_thanks=int((rid, uid) in thx_set),
            tie_strength=0.0,
            netpos=0.0,
            cluster_id=-1,
        )
        ev.validate()
        events.append(ev)
        eid += 1

    df = events_to_frame(events)
    if out_path:
        df.to_parquet(out_path, index=False)

    acted = (df["action"] != "none").mean() if len(df) else 0
    print(f"[ingest_real] events={len(df)} from {visits.shape[0]} visits")
    print(f"  acted={acted:.1%} non-acted={1-acted:.1%}")
    print(f"  action dist: {df['action'].value_counts().to_dict()}")
    print(f"  ripple_depth は state に保持(実データのネットワーク文脈)")
    return df



## 4. エクスポータ: フィジカルAI（RL遷移 + Gym env）

In [ ]:
"""
exporters/physical_ai.py
========================================================
フィジカルAI(模倣学習 / オフラインRL)向けエクスポータ。

出力:
  - transitions.parquet : (state, action, reward, next_state, done)
  - env_stub.py         : Gymnasium 互換の最小 env(SB3 で読める)

設計:
  - 各 actor の時系列イベントを「軌跡(trajectory)」として連結
  - state = schema の state_features(リーク無し)
  - action = none/reply/share の離散3
  - reward = config の reward_* (share に最大報酬)
  - next_state = 同 actor の次イベントの state(無ければ done=1)
"""

import pandas as pd
import numpy as np


def physical_ai_export(df: pd.DataFrame, cfg: dict, out_dir: str):
    assert_no_leak(df)
    rcfg = cfg["exporters"]["physical_ai"]
    reward_map = {"none": rcfg["reward_none"],
                  "reply": rcfg["reward_reply"],
                  "share": rcfg["reward_share"]}
    scols = state_columns(df)

    # actor ごとに時系列ソートして軌跡化(ベクトル化)
    df = df.sort_values(["actor_id", "timestamp"]).reset_index(drop=True)
    S = df[scols].to_numpy(dtype=float)
    actor = df["actor_id"].to_numpy()
    act_id = df["action"].map({"none": 0, "reply": 1, "share": 2}).to_numpy()
    rewards = df["action"].map(reward_map).to_numpy()

    # next state = 次行。ただし actor が変わる境界は done=1, ns=0
    n = len(df)
    NS = np.zeros_like(S)
    done = np.ones(n, dtype=int)
    same_next = actor[:-1] == actor[1:]
    NS[:-1][same_next] = S[1:][same_next]
    done[:-1][same_next] = 0

    tdf = pd.DataFrame(S, columns=[f"s__{c}" for c in scols])
    tdf.insert(0, "actor_id", actor)
    tdf["action_id"] = act_id
    tdf["reward"] = rewards
    for j, c in enumerate(scols):
        tdf[f"ns__{c}"] = NS[:, j]
    tdf["done"] = done
    path = f"{out_dir}/transitions.parquet"
    tdf.to_parquet(path, index=False)

    # Gym env stub (SB3 互換)
    stub = f'''"""
Gymnasium-compatible offline env over YOROSIX transitions.
Usage with Stable-Baselines3 (offline / behavior cloning):
    from env_stub import YorosixOfflineEnv
    env = YorosixOfflineEnv("transitions.parquet")
"""
import gymnasium as gym
import numpy as np, pandas as pd
from gymnasium import spaces

STATE_DIM = {len(scols)}
N_ACTIONS = 3

class YorosixOfflineEnv(gym.Env):
    def __init__(self, parquet_path):
        self.df = pd.read_parquet(parquet_path).reset_index(drop=True)
        self.scols = [c for c in self.df.columns if c.startswith("s__")]
        self.observation_space = spaces.Box(-np.inf, np.inf, ({len(scols)},), np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.i = 0
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed); self.i = 0
        return self.df.loc[0, self.scols].values.astype(np.float32), {{}}
    def step(self, action):
        row = self.df.loc[self.i]
        reward = float(row["reward"]) if action == row["action_id"] else 0.0
        done = bool(row["done"]); self.i = min(self.i + 1, len(self.df) - 1)
        ns = self.df.loc[self.i, self.scols].values.astype(np.float32)
        return ns, reward, done, False, {{}}
'''
    with open(f"{out_dir}/env_stub.py", "w") as f:
        f.write(stub)

    return {
        "transitions": path, "env_stub": f"{out_dir}/env_stub.py",
        "n_transitions": len(tdf), "state_dim": len(scols),
        "n_actors": df.actor_id.nunique(),
    }


def physical_ai_sanity(df: pd.DataFrame, cfg: dict, out_dir: str):
    """学習可能性チェック: データのポリシー報酬 > ランダムポリシー報酬 か。"""
    tdf = pd.read_parquet(f"{out_dir}/transitions.parquet")
    data_reward = tdf["reward"].mean()
    # ランダム行動が一致したときだけ報酬(=env step の仕様)を近似
    rng = np.random.default_rng(0)
    rand_match = (rng.integers(0, 3, len(tdf)) == tdf["action_id"].values)
    rand_reward = (tdf["reward"].values * rand_match).mean()
    ok = data_reward > rand_reward
    return {"data_policy_reward": float(data_reward),
            "random_policy_reward": float(rand_reward),
            "learnable": bool(ok)}

## 5. エクスポータ: ソーシャルシミュレーション（エージェント方策 + ネットワーク）

In [ ]:
"""
exporters/social_sim.py
========================================================
ソーシャルシミュレーション(ABM / Mesa)向けエクスポータ。

出力:
  - agent_policies.json : 各エージェントの行動確率モデル係数(個人別)
  - init_network.graphml: 初期ネットワーク(クラスタ + bridge エッジ)
  - mesa_model.py       : Mesa 互換の最小モデル(政策を読んで時間発展)

設計:
  - 各エージェントの行動を、state→P(share) のロジスティック係数として要約
  - シミュレータはこの係数で確率的に行動を再生 → 生成元の統計を再現できる
"""

import json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LogisticRegression


def social_sim_export(df: pd.DataFrame, cfg: dict, out_dir: str, world=None):
    assert_no_leak(df)
    scols = state_columns(df)

    # 全体ポリシー(共有の基底)+ クラスタ別の補正を係数として保存
    X = df[scols].values
    y = (df["action"] == "share").astype(int).values
    base = LogisticRegression(max_iter=2000).fit(X, y)

    policies = {
        "feature_order": scols,
        "global": {"coef": base.coef_[0].tolist(), "intercept": float(base.intercept_[0])},
        "per_cluster": {},
    }
    for c, g in df.groupby("cluster_id"):
        if len(g) < 50 or g["action"].nunique() < 2:
            continue
        Xc = g[scols].values
        yc = (g["action"] == "share").astype(int).values
        if yc.sum() == 0 or yc.sum() == len(yc):
            continue
        m = LogisticRegression(max_iter=1000).fit(Xc, yc)
        policies["per_cluster"][int(c)] = {
            "coef": m.coef_[0].tolist(), "intercept": float(m.intercept_[0])}

    with open(f"{out_dir}/agent_policies.json", "w") as f:
        json.dump(policies, f, indent=2)

    # 初期ネットワーク
    G = nx.Graph()
    if world is not None:
        for c, members in world.cluster_members.items():
            members = list(members)
            for i in range(len(members)):
                G.add_node(int(members[i]), cluster=int(c))
                # クラスタ内リング + 一部ランダム
                G.add_edge(int(members[i]), int(members[(i + 1) % len(members)]))
        # bridge 同士を繋ぐ
        br = list(world.bridges)
        for i in range(len(br)):
            G.add_edge(int(br[i]), int(br[(i + 1) % len(br)]))
    else:
        # world 無し: イベントデータからネットワークを復元
        # actor を cluster でグルーピングしてクラスタ内リングを張る
        actor_cluster = df.groupby("actor_id")["cluster_id"].first().to_dict()
        from collections import defaultdict
        by_cluster = defaultdict(list)
        for a, c in actor_cluster.items():
            G.add_node(int(a), cluster=int(c))
            by_cluster[int(c)].append(int(a))
        for c, members in by_cluster.items():
            for i in range(len(members)):
                G.add_edge(members[i], members[(i + 1) % len(members)])
    nx.write_graphml(G, f"{out_dir}/init_network.graphml")

    # Mesa モデル stub
    stub = '''"""
Mesa-compatible ABM that replays YOROSIX agent policies.
    from mesa_model import YorosixModel
    model = YorosixModel("agent_policies.json", "init_network.graphml")
    for _ in range(100): model.step()
"""
import json, numpy as np, networkx as nx
import mesa

def _sigmoid(x): return 1/(1+np.exp(-x))

class YorosixAgent(mesa.Agent):
    def __init__(self, uid, model, coef, intercept, order):
        super().__init__(uid, model)
        self.coef = np.array(coef); self.b = intercept; self.order = order
    def act(self, state_vec):
        z = float(self.coef @ state_vec + self.b)
        return np.random.rand() < _sigmoid(z)

class YorosixModel(mesa.Model):
    def __init__(self, policies_path, graph_path):
        super().__init__()
        pol = json.load(open(policies_path))
        self.G = nx.read_graphml(graph_path)
        self.order = pol["feature_order"]
        self.g_coef = pol["global"]["coef"]; self.g_b = pol["global"]["intercept"]
        self.per_cluster = pol["per_cluster"]
        self.schedule = mesa.time.RandomActivation(self)
        for n in self.G.nodes():
            c = str(self.G.nodes[n].get("cluster", -1))
            p = self.per_cluster.get(c, {"coef": self.g_coef, "intercept": self.g_b})
            self.schedule.add(YorosixAgent(int(n), self, p["coef"], p["intercept"], self.order))
    def step(self):
        self.schedule.step()
'''
    with open(f"{out_dir}/mesa_model.py", "w") as f:
        f.write(stub)

    return {"policies": f"{out_dir}/agent_policies.json",
            "network": f"{out_dir}/init_network.graphml",
            "mesa_model": f"{out_dir}/mesa_model.py",
            "n_agents": G.number_of_nodes(),
            "n_cluster_policies": len(policies["per_cluster"])}


def social_sim_sanity(df: pd.DataFrame, cfg: dict, out_dir: str):
    """再現性チェック: ポリシーで再生した行動率が元データと近いか。"""
    pol = json.load(open(f"{out_dir}/agent_policies.json"))
    scols = pol["feature_order"]
    coef = np.array(pol["global"]["coef"]); b = pol["global"]["intercept"]
    X = df[scols].values
    p = 1 / (1 + np.exp(-(X @ coef + b)))
    rng = np.random.default_rng(0)
    sim_share_rate = (rng.random(len(p)) < p).mean()
    real_share_rate = (df["action"] == "share").mean()
    return {"real_share_rate": float(real_share_rate),
            "simulated_share_rate": float(sim_share_rate),
            "abs_diff": float(abs(sim_share_rate - real_share_rate)),
            "reproduced": bool(abs(sim_share_rate - real_share_rate) < 0.05)}

## 6. エクスポータ: BDT（個人ツイン + 多次元アクション予測 + 反実仮想）

In [ ]:
"""
exporters/bdt.py
========================================================
行動デジタルツイン(BDT)向けエクスポータ。

出力:
  - digital_twins.pkl : 個人別の「状況→行動分布」モデル群
  - twin_api.py       : twin_query(actor_id, context) を提供

設計:
  - 全体モデル(GBDT)を基底とし、個人の潜在パラメータ(sens_*, latent_coop)
    を state に含めることで「個人ごとに状況→行動分布」を表現
  - 1 actor = 1 twin。context を入れると P(none/reply/share) を返す
  - 反実仮想クエリ対応(urgency を変えたら行動分布がどう動くか)
"""

import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier


def bdt_export(df: pd.DataFrame, cfg: dict, out_dir: str):
    assert_no_leak(df)
    scols = state_columns(df)

    # 多クラス(none/reply/share)を1つの基底モデルで学習。
    # 個人差は state に含まれる sens_*/latent_coop が担う = ツインの個性。
    X = df[scols].values
    y = df["action_id"].values  # 0/1/2
    base = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_depth=8,
        l2_regularization=1.0).fit(X, y)

    # --- 多次元アクション予測ヘッド(行動した人のみで学習) ---
    # 「行動するか」だけでなく「どんな強度・労力の返信か」まで返すツイン
    from sklearn.ensemble import HistGradientBoostingRegressor
    acted = df[df["action"] != "none"]
    intensity_head = effort_head = speed_head = None
    if len(acted) >= 50:
        Xa = acted[scols].values
        intensity_head = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=6).fit(Xa, acted["act_intensity"].values)
        effort_head = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=6).fit(Xa, acted["act_effort"].values)
        speed_head = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=6).fit(Xa, acted["act_speed"].values)

    # 個人の固定属性テーブル(twin の素性)
    fixed_cols = [c for c in scols if c.startswith("state__u_")]
    twin_profiles = (df.groupby("actor_id")[fixed_cols].first().to_dict("index"))

    bundle = {
        "base_model": base,
        "intensity_head": intensity_head,
        "effort_head": effort_head,
        "speed_head": speed_head,
        "feature_order": scols,
        "fixed_cols": fixed_cols,
        "twin_profiles": twin_profiles,
        "classes": ["none", "reply", "share"],
    }
    with open(f"{out_dir}/digital_twins.pkl", "wb") as f:
        pickle.dump(bundle, f)

    api = '''"""
BDT query API (multi-dimensional action).
    from twin_api import TwinRegistry
    reg = TwinRegistry("digital_twins.pkl")
    reg.query(actor_id=42, context={"req_urgency":0.8,"tie_strength":1.0,...})
      -> {"action_dist": {none,reply,share}, "if_acted": {intensity,effort,speed}}
    reg.counterfactual(actor_id=42, base_context={...}, urgency_delta=0.3)
"""
import pickle, numpy as np

class TwinRegistry:
    def __init__(self, path):
        b = pickle.load(open(path, "rb"))
        self.m = b["base_model"]; self.order = b["feature_order"]
        self.fixed = b["fixed_cols"]; self.prof = b["twin_profiles"]
        self.classes = b["classes"]
        self.ih = b.get("intensity_head"); self.eh = b.get("effort_head")
        self.sh = b.get("speed_head")
    def _vec(self, actor_id, context):
        prof = self.prof.get(actor_id, {})
        row = []
        for c in self.order:
            if c in prof: row.append(prof[c])
            elif c.replace("state__","") in context:
                row.append(context[c.replace("state__","")])
            elif c in context: row.append(context[c])
            else: row.append(0.0)
        return np.array(row).reshape(1, -1)
    def query(self, actor_id, context):
        x = self._vec(actor_id, context)
        p = self.m.predict_proba(x)[0]
        out = {"action_dist": dict(zip(self.classes, p.tolist()))}
        # 行動したらどんな返信か(強度・労力・速さ)
        if self.ih is not None:
            out["if_acted"] = {
                "intensity": float(self.ih.predict(x)[0]),
                "effort": float(self.eh.predict(x)[0]),
                "speed": float(self.sh.predict(x)[0]),
            }
        return out
    def counterfactual(self, actor_id, base_context, urgency_delta=0.3):
        cf = dict(base_context)
        cf["req_urgency"] = min(1.0, cf.get("req_urgency", 0.5) + urgency_delta)
        return {"base": self.query(actor_id, base_context),
                "counterfactual": self.query(actor_id, cf)}
'''
    with open(f"{out_dir}/twin_api.py", "w") as f:
        f.write(api)

    return {"twins": f"{out_dir}/digital_twins.pkl",
            "api": f"{out_dir}/twin_api.py",
            "n_twins": len(twin_profiles), "state_dim": len(scols)}


def bdt_sanity(df: pd.DataFrame, cfg: dict, out_dir: str):
    """反実仮想チェック: ツインに urgency+Δ を入れたら share 確率が上がるか。"""
    import pickle
    b = pickle.load(open(f"{out_dir}/digital_twins.pkl", "rb"))
    m = b["base_model"]; order = b["feature_order"]
    delta = cfg["counterfactual"]["urgency_delta"]
    uidx = order.index("state__req_urgency") if "state__req_urgency" in order else None
    if uidx is None:
        return {"counterfactual_supported": False}
    X = df[order].values.copy()
    p_base = m.predict_proba(X)[:, 2].mean()       # class 2 = share
    Xcf = X.copy(); Xcf[:, uidx] = np.clip(Xcf[:, uidx] + delta, 0, 1)
    p_cf = m.predict_proba(Xcf)[:, 2].mean()
    return {"share_prob_base": float(p_base),
            "share_prob_cf": float(p_cf),
            "lift": float(p_cf - p_base),
            "direction_correct": bool(p_cf > p_base)}


# === ノートブック内で使える TwinRegistry（twin_api.py と同じ実装） ===
class TwinRegistry:
    def __init__(self, path):
        b = pickle.load(open(path, "rb"))
        self.m = b["base_model"]; self.order = b["feature_order"]
        self.fixed = b["fixed_cols"]; self.prof = b["twin_profiles"]
        self.classes = b["classes"]
        self.ih = b.get("intensity_head"); self.eh = b.get("effort_head")
        self.sh = b.get("speed_head")
    def _vec(self, actor_id, context):
        prof = self.prof.get(actor_id, {})
        row = []
        for c in self.order:
            if c in prof: row.append(prof[c])
            elif c.replace("state__", "") in context:
                row.append(context[c.replace("state__", "")])
            elif c in context: row.append(context[c])
            else: row.append(0.0)
        return np.array(row).reshape(1, -1)
    def query(self, actor_id, context):
        x = self._vec(actor_id, context)
        p = self.m.predict_proba(x)[0]
        out = {"action_dist": dict(zip(self.classes, [round(v,3) for v in p.tolist()]))}
        if self.ih is not None:
            out["if_acted"] = {
                "intensity": round(float(self.ih.predict(x)[0]), 3),
                "effort": round(float(self.eh.predict(x)[0]), 3),
                "speed": round(float(self.sh.predict(x)[0]), 3),
            }
        return out
    def counterfactual(self, actor_id, base_context, urgency_delta=0.3):
        cf = dict(base_context)
        cf["req_urgency"] = min(1.0, cf.get("req_urgency", 0.5) + urgency_delta)
        return {"base": self.query(actor_id, base_context),
                "counterfactual": self.query(actor_id, cf)}

## 7. 検証スイート（5プロトコル + エクスポータ健全性）

In [ ]:
"""
validation/protocols.py
========================================================
データ品質の5プロトコル + 各エクスポータの健全性。
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score


def run_data_protocols(df: pd.DataFrame):
    assert_no_leak(df)
    scols = state_columns(df)
    df = df.copy()
    df["label_act"] = (df["action"] != "none").astype(int)
    y = df["label_act"].values

    rng = np.random.default_rng(7)
    reqs = df["request_id"].unique(); rng.shuffle(reqs)
    te = set(reqs[:len(reqs)//5])
    trm = ~df["request_id"].isin(te).values
    tem = df["request_id"].isin(te).values

    sc = StandardScaler(); X = sc.fit_transform(df[scols])
    lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[trm], y[trm])
    auc = roc_auc_score(y[tem], lr.predict_proba(X[tem])[:, 1])

    # P1
    bal = y.mean()
    p1 = bal < 0.5
    # P3 same_cluster 単独
    sccol = "state__same_cluster"
    if sccol in df.columns:
        Xs = sc.fit_transform(df[[sccol]])
        a = roc_auc_score(y[tem], LogisticRegression(max_iter=500, class_weight="balanced")
                          .fit(Xs[trm], y[trm]).predict_proba(Xs[tem])[:, 1])
    else:
        a = 0.5
    p3 = a < 0.65
    # P2 文脈が効いてるか(urgency感受性 or topic_fit)
    coef = dict(zip(scols, lr.coef_[0]))
    p2 = (abs(coef.get("state__req_urgency", 0)) + abs(coef.get("state__u_sens_urgency", 0))
          + abs(coef.get("state__topic_fit", 0))) > 0.05

    return {
        "P1_nonaction_majority": {"action_rate": float(bal), "pass": bool(p1)},
        "P2_context_effective": {"pass": bool(p2)},
        "P3_label_independent": {"same_cluster_auc": float(a), "pass": bool(p3)},
        "full_model_auc": float(auc),
    }


def run_exporter_checks(df, cfg, out_dir):
    return {
        "physical_ai": physical_ai_sanity(df, cfg, out_dir),
        "social_sim": social_sim_sanity(df, cfg, out_dir),
        "bdt": bdt_sanity(df, cfg, out_dir),
    }

## 8. 実行: 設定

In [ ]:
CONFIG = {
    "world": {"n_members": 5000, "n_posts": 2000, "seed": 42, "n_clusters": 20, "n_topics": 6},
    "generation": {"beta0_share": -5.0, "beta0_reply": -4.6, "w_latent_coop": 1.8,
                   "w_urgency": 2.0, "w_tie": 1.4, "w_time": 0.7, "w_load": 0.9,
                   "w_netpos": 1.0, "w_topic": 1.1, "noise_sd": 0.35},
    "counterfactual": {"enabled": True, "urgency_delta": 0.3},
    "output": {"dir": "yorosix_output", "format": "parquet"},
    "exporters": {
        "physical_ai": {"reward_share": 1.0, "reward_reply": 0.5, "reward_none": 0.0},
        "social_sim": {"framework": "mesa"},
        "bdt": {"model": "gradient_boosting"},
    },
}
import os
os.makedirs(CONFIG["output"]["dir"], exist_ok=True)

## 9. 合成データ生成 → 標準スキーマ

In [ ]:
world = YorosixWorld(CONFIG)
events = world.generate()
df = events_to_frame(events)
assert_no_leak(df)
df.to_parquet(f"{CONFIG['output']['dir']}/events.parquet", index=False)
acted = (df.action != "none").mean()
print(f"events={len(df)}  acted={acted:.1%}  non-acted={1-acted:.1%}")
print("action dist:", df.action.value_counts().to_dict())
df[["action","act_intensity","act_effort","act_speed","act_content_topic"]].head()

## 10. 多次元アクションが文脈依存か確認

In [ ]:
acted = df[df.action != "none"]
hi = acted[acted["state__req_urgency"]>0.7]["act_intensity"].mean()
lo = acted[acted["state__req_urgency"]<0.3]["act_intensity"].mean()
print(f"intensity 高緊急={hi:.3f}  低緊急={lo:.3f}  差={hi-lo:+.3f}")
print(f"effort reply={acted[acted.action=='reply'].act_effort.mean():.3f}  "
      f"share={acted[acted.action=='share'].act_effort.mean():.3f}")

## 11. 3用途へエクスポート

In [ ]:
d = CONFIG["output"]["dir"]
r1 = physical_ai_export(df, CONFIG, d)
r2 = social_sim_export(df, CONFIG, d, world=world)
r3 = bdt_export(df, CONFIG, d)
print("physical_ai:", r1["n_transitions"], "transitions")
print("social_sim :", r2["n_agents"], "agents")
print("bdt        :", r3["n_twins"], "twins")

## 12. 検証

In [ ]:
# 反実仮想世界も生成（検証で使う）
if CONFIG["counterfactual"]["enabled"]:
    cf_events = world.generate(CONFIG["counterfactual"]["urgency_delta"])
    cf_df = events_to_frame(cf_events)
    real_rate = (df.action != "none").mean()
    cf_rate = (cf_df.action != "none").mean()
    json.dump({"urgency_delta": CONFIG["counterfactual"]["urgency_delta"],
               "real_act_rate": float(real_rate), "cf_act_rate": float(cf_rate)},
              open(f"{CONFIG['output']['dir']}/counterfactual_summary.json", "w"))
    print(f"反実仮想: 行動率 {real_rate:.1%} -> {cf_rate:.1%}")

data = run_data_protocols(df)
exp = run_exporter_checks(df, CONFIG, CONFIG["output"]["dir"])
print("P1 非行動多数派:", data["P1_nonaction_majority"]["pass"])
print("P2 文脈が効く  :", data["P2_context_effective"]["pass"])
print("P3 逆算不能    :", data["P3_label_independent"]["pass"],
      f"(same_cluster AUC {data['P3_label_independent']['same_cluster_auc']:.3f})")
print("full-model AUC :", round(data["full_model_auc"], 4))
print("physical_ai 学習可能:", exp["physical_ai"]["learnable"])
print("social_sim 再現    :", exp["social_sim"]["reproduced"])
print("bdt 反実仮想方向   :", exp["bdt"]["direction_correct"])

## 13. BDTツインに多次元アクションを問い合わせる

同じ人でも文脈で「行動の中身」が変わることを確認。

In [ ]:
d = CONFIG["output"]["dir"]
reg = TwinRegistry(f"{d}/digital_twins.pkl")
aid = list(pickle.load(open(f"{d}/digital_twins.pkl", "rb"))["twin_profiles"].keys())[0]

print(f"=== Twin actor {aid} ===")
print("低緊急・弱紐帯:")
print("  ", reg.query(aid, {"req_urgency": 0.2, "tie_strength": 0.2}))
print("高緊急・強紐帯(同じ人):")
print("  ", reg.query(aid, {"req_urgency": 0.9, "tie_strength": 1.0}))
print("反実仮想(urgency+0.3):")
print("  ", reg.counterfactual(aid, {"req_urgency": 0.5, "tie_strength": 0.6}))

## 14.（任意）実データの取り込み

本番CSVがあるディレクトリを指定。合成と同じスキーマ→同じエクスポータが動く。

In [ ]:
# DATA_DIR = "/path/to/yorosix_exports"   # 本番CSVのディレクトリ
# real_df = ingest(DATA_DIR, f"{d}/events_real.parquet")
# physical_ai_export(real_df, CONFIG, d)
# social_sim_export(real_df, CONFIG, d)
# bdt_export(real_df, CONFIG, d)
print("実データを使う場合は上のコメントを外して DATA_DIR を指定")